# Extração do texto e limpeza

Aqui tiramos o texto das fontes em `data/raw/`, fazemos a limpeza e gravamos o resultado em `data/interim/`.

| Arquivo | Conteúdo |
|---|---|
| `sumulas_limpas.jsonl` | Número, marca, enunciado, citação e histórico de cada súmula |
| `temas_limpos.jsonl` | Tese, questão e demais campos de texto de cada precedente |

A limpeza só corrige problemas de formato. Acentos, stopwords e maiúsculas continuam como estão, para que o texto exibido fique fiel à fonte. Situação, vigência e links ficam para o notebook 03.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
INTERIM = ROOT / "data/interim"
INTERIM.mkdir(parents=True, exist_ok=True)


RE_HIFEN = re.compile(r"(?<=[a-zà-ú])- (?=[a-zà-ú])")  # "estado- membro"; antes de sigla o hífen pode ser separador


def limpar(texto: str) -> str:
    return RE_HIFEN.sub("-", re.sub(r"\s+", " ", texto)).strip()


def gravar_jsonl(registros: list[dict], destino: Path) -> None:
    with destino.open("w", encoding="utf-8") as arquivo:
        for registro in registros:
            arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")
    print(f"{destino.relative_to(ROOT)}: {len(registros)} registros")

## Extração das súmulas

Cada súmula começa com um cabeçalho `SÚMULA n [(SÚMULA CANCELADA)] VEJA MAIS`, e o texto dela vai até o cabeçalho seguinte.

In [2]:
leitor = PdfReader(RAW / "VerbetesSTJ.pdf")
print("Gerado em:", leitor.metadata.get("/CreationDate"), "| páginas:", len(leitor.pages))
texto_pdf = "\n".join(pagina.extract_text() for pagina in leitor.pages)

RE_CABECALHO = re.compile(r"^\s*●?\s*SÚMULA (\d+)(?: \(([^)]*)\))?\s*VEJA MAIS\s*$", re.M)

cabecalhos = list(RE_CABECALHO.finditer(texto_pdf))
corpos = {
    int(atual[1]): (atual[2], texto_pdf[atual.end() : proximo.start() if proximo else len(texto_pdf)])
    for atual, proximo in zip(cabecalhos, [*cabecalhos[1:], None])
}
print("Súmulas encontradas:", len(corpos))

Gerado em: D:20251205111346-03'00' | páginas: 93


Súmulas encontradas: 676


## Limpeza das súmulas

O corpo de cada súmula traz o enunciado, a citação entre parênteses e, em alguns casos, um histórico com a `REDAÇÃO ANTERIOR`. O enunciado em vigor é só o que vem antes da citação. O histórico é guardado à parte para que a redação antiga não entre no índice.

In [3]:
RE_RODAPE = re.compile(r"scon\.stj\.jus\.br/\S*\s*\d+(?:/\d+)?")
RE_CITACAO = re.compile(r"\((?:SÚMULA \d+,\s*)?[A-ZÇÃÉÍÓÚ ]+,\s*julgad[oa] em", re.I)
RE_HISTORICO = re.compile(r"SÚMULA (?:CANCELADA|REVOGADA|ALTERADA|REVISADA)\s*:|REDAÇÃO ANTERIOR")


def separar_corpo(corpo: str) -> tuple[str, str, str]:
    """Separa o corpo em (enunciado, citação, histórico)."""
    corpo = limpar(RE_RODAPE.sub(" ", corpo))
    historico = RE_HISTORICO.search(corpo)
    trecho_atual = corpo[: historico.start()].strip() if historico else corpo
    resto = corpo[historico.start() :].strip() if historico else ""
    citacao = RE_CITACAO.search(trecho_atual)
    if citacao:
        return trecho_atual[: citacao.start()].strip(), trecho_atual[citacao.start() :].strip(), resto
    return trecho_atual, "", resto


sumulas = []
for numero, (marca, corpo) in corpos.items():
    enunciado, citacao, historico = separar_corpo(corpo)
    sumulas.append({"numero": numero, "marca": marca, "enunciado": enunciado, "citacao": citacao, "historico": historico})

numeros = {s["numero"] for s in sumulas}
assert numeros == set(range(1, max(numeros) + 1)), "numeração de súmulas com lacuna"
assert all(len(s["enunciado"]) > 30 for s in sumulas), "enunciado vazio ou truncado"
assert not any(re.search(r"REDAÇÃO ANTERIOR|julgad[oa] em \d|VEJA MAIS", s["enunciado"]) for s in sumulas)
print("Súmulas:", len(sumulas), "| última:", max(numeros), "|", Counter(s["marca"] for s in sumulas))
print("Sem citação no PDF:", [s["numero"] for s in sumulas if not s["citacao"]])

Súmulas: 676 | última: 676 | Counter({None: 641, 'SÚMULA CANCELADA': 29, 'SÚMULA ALTERADA': 5, 'SÚMULA REVOGADA': 1})
Sem citação no PDF: [217, 152]


Veja como fica uma súmula que tem histórico, antes e depois da limpeza.

In [4]:
por_numero = {s["numero"]: s for s in sumulas}
print("ANTES:\n", corpos[630][1])
print("\nDEPOIS:")
for campo in ("enunciado", "citacao", "historico"):
    print(f"- {campo}: {por_numero[630][campo]}")

ANTES:
 
A incidência da atenuante da confissão espontânea no crime de tráfico ilícito de entorpecentes
quando o acusado admitir a posse ou a propriedade para uso próprio, negando a prática do tráfico
de drogas, deve ocorrer em proporção inferior à que seria devida no caso de confissão plena.
(TERCEIRA SEÇÃO, julgado em 10/9/2025, DJEN de 2/12/2025) SÚMULA REVISADA: A Terceira Seção,
na sessão ordinária de 10/09/2025, ao julgar o REsp 2.001.973/RS (tema repetitivo 1194), deliberou
pela REVISÃO do enunciado da Súmula 630. REDAÇÃO ANTERIOR: A incidência da atenuante da
confissão espontânea no crime de tráfico ilícito de entorpecentes exige o reconhecimento da
traficância pelo acusado, não bastando a mera admissão da posse ou propriedade para uso próprio.
(SÚMULA 630, TERCEIRA SEÇÃO, julgado em 24/4/2019, DJe de 29/4/2019)


DEPOIS:
- enunciado: A incidência da atenuante da confissão espontânea no crime de tráfico ilícito de entorpecentes quando o acusado admitir a posse ou a propriedade 

## Leitura e limpeza dos precedentes

O CSV já vem separado em colunas, então o trabalho é menor. Juntamos espaços e quebras de linha, removemos os duplicados e descartamos as linhas sem tese e sem questão, porque nelas não há o que indexar.

In [5]:
temas_csv = pd.read_csv(RAW / "temas.csv", dtype=str, encoding="utf-8-sig").fillna("")
print("Linhas no CSV:", len(temas_csv))

temas = []
for _, linha in temas_csv.drop_duplicates(["tipoPrecedente", "numeroPrecedente"]).iterrows():
    tese, questao = limpar(linha["teseFirmada"]), limpar(linha["questaoSubmetidaAJulgamento"])
    if not tese and not questao:
        continue
    temas.append(
        {
            "tipo": linha["tipoPrecedente"],
            "numero": int(linha["numeroPrecedente"]),
            "situacao": linha["situacao"],
            "tese": tese,
            "questao": questao,
            "orgao": linha["orgaoJulgador"],
            "entendimento_anterior": limpar(linha["entendimentoAnterior"]),
            "assuntos": limpar(linha["Assuntos"]),
        }
    )
print("Precedentes com texto:", len(temas))

Linhas no CSV: 2410
Precedentes com texto: 2369


## Auditoria

Esta parte procura restos de extração nos textos que vão para o índice e interrompe a execução se encontrar algum.

In [6]:
RESIDUOS = {
    "HTML": r"</?[a-zA-Z][^>]*>|&[a-z]+;|&#\d+;",
    "caractere de controle": r"[\x00-\x08\x0b-\x1f\x7f�­]",
    "encoding corrompido": r"Ã[\x80-\xbf§£¡©ª³µ]|Â[ºª]",
    "rodapé do PDF": r"scon\.stj|VEJA MAIS",
    "hífen com espaço": r"[a-zà-ú]- [a-zà-ú]",
}
textos = [s["enunciado"] for s in sumulas] + [t[campo] for t in temas for campo in ("tese", "questao")]
encontrados = {
    nome: [re.search(padrao, texto)[0] for texto in textos if re.search(padrao, texto)]
    for nome, padrao in RESIDUOS.items()
}
for nome, casos in encontrados.items():
    print(f"{nome}: {len(casos)} {casos[:5]}")
assert not any(encontrados.values()), "resíduo de extração nos textos"

HTML: 0 []
caractere de controle: 0 []
encoding corrompido: 0 []
rodapé do PDF: 0 []
hífen com espaço: 0 []


## Gravação

In [7]:
gravar_jsonl(sumulas, INTERIM / "sumulas_limpas.jsonl")
gravar_jsonl(temas, INTERIM / "temas_limpos.jsonl")

data/interim/sumulas_limpas.jsonl: 676 registros
data/interim/temas_limpos.jsonl: 2369 registros
